In [7]:
from interpret.glassbox import ExplainableBoostingClassifier
from interpret import show
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

df = pd.read_csv('../data/processed/NY_SLI_ATTOM_EXTENDED.csv')
df_numeric_data = df.drop(columns=[
    'Lead Gooseneck, Pigtail or Connector Currently Present',
    'Current Public Side SL Material',
    'Was Public SL Material Ever Previously Lead',
    'Public SL Material Verification Method',
    'Customer SL Material',
    'Lead Solder Present',
    'Customer SL Size',
    'SL Category', 
    'Note', 
    'Current SL Material Category', 'Lead Connector Present Category',
    'Current Public Side SL Material Category',
    'Was Public SL Material Ever Previously Lead Category',
    'Public SL Size Numeric', 
    'attomID',
    'Service Line Locality',
    'Street Address',
    'State',
    'Public SL Size',
    'Customer SL Material Verification Method',
    'POU or POE Treatment Present',
    'Building Type',
    'prop_class',
    'size_ind',
    'Public SL Installation Year',
    'Public SL Material Verification Method Category',
    'Public SL Size Category',
    'Zip Code',
    'Customer SL Material Category',
])
print(df_numeric_data.columns)
def safe_float(x):
    try:
        return float(x)
    except (ValueError, TypeError):
        return False
def safe_install_year(x) -> bool:
    return safe_float(x) > 1500 and safe_float(x) < 2025

def condition_numeric(x):
    if x == 'NaN' or type(x) == float: # only if nan
        return 0
    if x == 'FAIR':
        return 1
    if x == 'AVERAGE':
        return 2
    if x == 'GOOD':
        return 3
    if x == 'EXCELLENT':
        return 4
df_numeric_data[['Longitude', 'Latitude']] = (
    df_numeric_data['Location']
    .str.extract(r'POINT\s*\(\s*([-.\d]+)\s+([-.\d]+)\s*\)')
    .astype(float)
)
df_numeric_data = df_numeric_data.drop(columns='Location')
df_numeric_data = df_numeric_data[df_numeric_data['living_size'] < 10_000]
df_numeric_data['construction_condition'] = df_numeric_data['construction_condition'].apply(condition_numeric)
df_numeric_data['Public SL Installation or Replacement Date'] = df_numeric_data.apply(lambda row: row['year_built'] if not safe_install_year(row['Public SL Installation or Replacement Date']) else float(row['Public SL Installation or Replacement Date']), axis=1 )
for col in df_numeric_data:
    if col != 'SL Category Cleaned':
        df_numeric_data[col] = df_numeric_data[col].apply(float)

len(df_numeric_data[df_numeric_data['SL Category Cleaned'] == 'Lead'])

Index(['Unnamed: 0', 'Public SL Installation or Replacement Date',
       'Customer SL Installation or Replacement Date', 'Location',
       'SL Category Cleaned', 'lot_size2', 'year_built', 'living_size', 'beds',
       'baths_total',
       ...
       'avg_Sep_Precip_In', 'avg_Oct_Precip_In', 'avg_Nov_Precip_In',
       'avg_Dec_Precip_In', 'weather_Index', 'earthquake_Index', 'hail_Index',
       'hurricane_Index', 'tornado_Index', 'wind_Index'],
      dtype='object', length=643)


318

In [8]:
y = df_numeric_data['SL Category Cleaned']
print(len(df_numeric_data.columns))
df_numeric_data = df_numeric_data.drop(columns='SL Category Cleaned')
X = df_numeric_data.to_numpy()
print(len(X[0]))
# X = np.hstack([X[:, 0:2], X[:, 3:]])
print(len(X[0]))

y = y[~np.isnan(X).any(axis=1)]
X = X[~np.isnan(X).any(axis=1)]
X
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

pca = PCA(n_components=10)
X_train_reduced = pca.fit_transform(X_train_scaled)
X_test_reduced = pca.transform(X_test_scaled)

644
643
643


In [ ]:
ebm = ExplainableBoostingClassifier()
ebm.fit(X_train_scaled, y_train)

In [10]:
ebm_global = ebm.explain_global()
show(ebm_global)

<!-- http://127.0.0.1:7001/131330765980352/ -->